# BERT Intent Recognition: Training and Evaluation

Fine-tune `bert-base-uncased` to classify customer-support queries into 27 intents, then evaluate on a held-out test set.

Runs on Colab (GPU runtime recommended) or locally. Upload `Customer_Service_Training_Dataset.csv` next to the notebook, or place it in `data/`.

## 1. Setup

In [ ]:
# Colab: install what is missing. Skip this cell if your environment already has these.
%pip install -q "transformers==4.52.4" datasets accelerate scikit-learn pandas matplotlib seaborn

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.metrics.pairwise import linear_kernel
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

SEED = 42
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
OUT_DIR = "bert_intent_model"
set_seed(SEED)

CANDIDATES = [Path("data/Customer_Service_Training_Dataset.csv"),
              Path("../data/Customer_Service_Training_Dataset.csv"),
              Path("Customer_Service_Training_Dataset.csv")]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
assert DATA_PATH is not None, "Dataset CSV not found. Upload it or put it in data/."
print("Using", DATA_PATH, "| GPU available:", torch.cuda.is_available())

## 2. Data

Select the text and label columns by name when possible. The original notebook picked them by position, which breaks silently if the column order changes.

In [ ]:
raw = pd.read_csv(DATA_PATH)
if {"instruction", "intent"}.issubset(raw.columns):
    TEXT_COL, LABEL_COL = "instruction", "intent"
else:
    TEXT_COL, LABEL_COL = raw.columns[1], raw.columns[3]   # fallback: same positions as before
print(f"text column = {TEXT_COL!r}, label column = {LABEL_COL!r}")

df = raw[[TEXT_COL, LABEL_COL]].rename(columns={TEXT_COL: "text", LABEL_COL: "intent"}).dropna()
df["text"] = df["text"].astype(str).str.strip()

conflicts = int((df.assign(k=df["text"].str.lower()).groupby("k")["intent"].nunique() > 1).sum())
n_before = len(df)
df = df[~df["text"].str.lower().duplicated()].reset_index(drop=True)
print(f"{n_before} rows -> {len(df)} after dropping duplicate queries | queries with conflicting labels: {conflicts}")
print(df["intent"].nunique(), "intents")
df.head()

In [ ]:
df["intent"].value_counts().sort_values().plot(kind="barh", figsize=(7, 9), title="Queries per intent")
plt.tight_layout(); plt.show()
print(df["text"].str.split().str.len().describe().round(1))

## 3. Stratified train / validation / test split

In [ ]:
class_names = sorted(df["intent"].unique())
label2id = {c: i for i, c in enumerate(class_names)}
id2label = {i: c for c, i in label2id.items()}
df["label"] = df["intent"].map(label2id)

train_df, rest_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(rest_df, test_size=0.5, stratify=rest_df["label"], random_state=SEED)
train_df, val_df, test_df = (d.reset_index(drop=True) for d in (train_df, val_df, test_df))
print(len(train_df), "train |", len(val_df), "validation |", len(test_df), "test")

In [ ]:
# How many test queries have a near-duplicate in the training set?
# Template-generated datasets often do, which inflates scores compared with real user messages.
vec = TfidfVectorizer(ngram_range=(1, 2)).fit(train_df["text"])
nearest = linear_kernel(vec.transform(test_df["text"]), vec.transform(train_df["text"])).max(axis=1)
near_dup_share = float((nearest >= 0.9).mean())
print(f"{near_dup_share:.1%} of test queries have a near-duplicate (cosine >= 0.9) in the training set")

## 4. Tokenise

Padding is applied per batch by `DataCollatorWithPadding`, so short queries are not padded to 128 tokens.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(frame):
    ds = Dataset.from_pandas(frame[["text", "label"]])
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["text"])

train_ds, val_ds, test_ds = to_dataset(train_df), to_dataset(val_df), to_dataset(test_df)

## 5. Train

Label names are stored inside the model config (`id2label`), so the app does not need a separate `label_encoder.pkl`. The best epoch is chosen on validation macro-F1, never on the test set.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(class_names), id2label=id2label, label2id=label2id)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "macro_f1": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",          # called evaluation_strategy in transformers < 4.41
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)

In [ ]:
# needs_model
trainer.train()

## 6. Evaluate on the held-out test set

In [ ]:
# needs_model
pred_out = trainer.predict(test_ds)

In [ ]:
logits = pred_out.predictions
e = np.exp(logits - logits.max(axis=1, keepdims=True))
probs = e / e.sum(axis=1, keepdims=True)

y_true, y_pred, conf = pred_out.label_ids, probs.argmax(axis=1), probs.max(axis=1)
test_acc = float(accuracy_score(y_true, y_pred))
test_f1 = float(f1_score(y_true, y_pred, average="macro"))
print(f"Test accuracy: {test_acc:.4f} | Test macro-F1: {test_f1:.4f} | n = {len(y_true)}")
print()
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

In [ ]:
cm = confusion_matrix(y_true, y_pred, normalize="true")
plt.figure(figsize=(13, 11))
sns.heatmap(cm, xticklabels=class_names, yticklabels=class_names, cmap="Blues", vmin=0, vmax=1, cbar_kws={"label": "Share of true class"})
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Normalised confusion matrix (test set)")
plt.xticks(rotation=90); plt.tight_layout(); plt.show()

## 7. Error analysis

In [ ]:
raw_cm = confusion_matrix(y_true, y_pred)
np.fill_diagonal(raw_cm, 0)
pairs = [(class_names[i], class_names[j], int(raw_cm[i, j])) for i, j in zip(*np.nonzero(raw_cm))]
print("Most confused intent pairs:")
display(pd.DataFrame(sorted(pairs, key=lambda p: -p[2])[:10], columns=["true", "predicted", "count"]))

errors = test_df.assign(pred=[class_names[i] for i in y_pred], confidence=conf.round(3))
errors = errors[errors["intent"] != errors["pred"]]
print(f"{len(errors)} misclassified of {len(test_df)}")
display(errors[["text", "intent", "pred", "confidence"]].head(15))

## 8. Confidence and out-of-scope behaviour

A closed-set classifier must pick one of 27 intents even for unrelated text. The confidence threshold lets the app decline instead. Use this table to choose it.

In [ ]:
rows = []
for t in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9]:
    answered = conf >= t
    rows.append({"threshold": t,
                 "share answered": round(float(answered.mean()), 3),
                 "accuracy on answered": round(float((y_pred[answered] == y_true[answered]).mean()), 3) if answered.any() else np.nan})
threshold_table = pd.DataFrame(rows).set_index("threshold")
threshold_table

In [ ]:
# needs_model
probes = ["what is the weather today", "tell me a joke", "who won the cricket match",
          "how do I bake a chocolate cake", "asdf qwerty", "I love this phone"]
enc = tokenizer(probes, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(trainer.model.device)
with torch.no_grad():
    lg = trainer.model(**enc).logits.cpu().numpy()
pp = np.exp(lg - lg.max(axis=1, keepdims=True)); pp /= pp.sum(axis=1, keepdims=True)
pd.DataFrame({"query": probes, "predicted intent": [class_names[i] for i in pp.argmax(1)], "confidence": pp.max(1).round(3)})

## 9. Save

In [ ]:
# needs_model
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("Saved model and tokenizer to", OUT_DIR)

In [ ]:
metrics = {
    "test_accuracy": round(test_acc, 4),
    "test_macro_f1": round(test_f1, 4),
    "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
    "n_intents": len(class_names),
    "near_duplicate_share_of_test": round(near_dup_share, 4),
    "threshold_table": threshold_table.reset_index().to_dict(orient="records"),
}
Path("metrics.json").write_text(json.dumps(metrics, indent=2))
metrics

### Publish the model

The weights are about 420 MB, too large for GitHub, so host them on the Hugging Face Hub and point the app at them with `MODEL_ID`.

```python
from huggingface_hub import notebook_login
notebook_login()
model.push_to_hub("your-username/bert-intent-ecommerce")
tokenizer.push_to_hub("your-username/bert-intent-ecommerce")
```